# Stage 5 — Training the fixes on Kaggle (E2, E4)

Run this notebook **twice** with *Save Version → Save & Run All*: once with `RUN = "E2"` and once
with `RUN = "E4"` (in parallel if Kaggle allows two GPU sessions, otherwise one after the other).
Requirements: GPU T4, Internet on, RSNA competition attached as input.

- **E2:** lung-only input (convex hull of each lung, dilated 5 px; suspect masks replaced by the
  mean hull), trained and evaluated that way. 3 seeds.
- **E4:** adversarial deconfounding (score independent of view given the label). λ is chosen on
  validation from {0.3, 1, 3} with seed 0; λ = 0 is a reproduction of E1 with the same code. Rule:
  among λ > 0 whose validation within-view AUC is at most 0.01 below λ = 0, take the one with the
  lowest validation leakage. Then seeds 1 and 2 with that λ.
- All new runs select the checkpoint on **validation within-view AUC** (mean of AP and PA AUC),
  because overall AUC rewards the view shortcut.

In [ ]:
RUN = "E2"   # "E2" or "E4"

In [ ]:
from pathlib import Path
import sys
REPO = Path("/kaggle/working/xai-cxr-shortcuts")
if REPO.exists():
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/xabieralbizu2/V1_XRay_xAI {REPO}
!pip install -q torchxrayvision pydicom
sys.path.insert(0, str(REPO))
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import pandas as pd
from src.data import REPO_ROOT, find_data_root, build_metadata, CLASSES_FILE
from src.preprocess import build_image_cache
from src.utils import get_device

DATA = REPO_ROOT / "data"
root = find_data_root()
device = get_device()
assign = pd.read_csv(REPO_ROOT / "splits" / "patient_splits.csv")
splits = build_metadata(root).merge(assign, on="patientId", how="inner")
if "class" not in splits.columns:
    cls = pd.read_csv(root / CLASSES_FILE).drop_duplicates("patientId")[["patientId", "class"]]
    splits = splits.merge(cls, on="patientId")
assert len(splits) == len(assign) == 26679, "splits do not match the local ones"

cache_path = DATA / "processed" / "images_224.npy"
if not cache_path.exists():
    build_image_cache(root, splits.patientId, cache_path)

In [ ]:
# Lung masks and hulls (only needed for E2; about 15 min on a T4)
from src.masks import build_mask_cache, build_hull_cache

mask_path = DATA / "processed" / "lung_masks_224.npy"
hull_path = DATA / "processed" / "lung_hulls_224.npy"
if RUN == "E2":
    if not mask_path.exists():
        build_mask_cache(cache_path, mask_path, device, batch_size=64)
    if not hull_path.exists():
        build_hull_cache(mask_path, hull_path)

In [ ]:
from src.train import TrainConfig, train_model

out_dir = Path("/kaggle/working/checkpoints")
common = dict(num_workers=4, select_on="within_view_auc")

if RUN == "E2":
    for seed in [0, 1, 2]:
        cfg = TrainConfig(experiment="E2", seed=seed, input_mask="lung", mask_path=str(hull_path), **common)
        train_model(cfg, cache_path, splits, out_dir, device)

elif RUN == "E4":
    # 1) lambda grid with seed 0 (lambda = 0 reproduces E1 with this code)
    grid = [0.0, 0.3, 1.0, 3.0]
    rows = []
    for lam in grid:
        cfg = TrainConfig(experiment=f"E4_lam{lam}", seed=0, adv_weight=lam, **common)
        h = train_model(cfg, cache_path, splits, out_dir, device)
        best = h.loc[h.val_within_view_auc.idxmax()]
        rows.append({"lambda": lam, "epoch": best.epoch, "val_auc": best.val_auc,
                     "val_within_view_auc": best.val_within_view_auc, "val_leakage": best.val_leakage})
    summary = pd.DataFrame(rows).set_index("lambda")
    summary.to_csv(out_dir / "e4_lambda_grid.csv")
    print(summary.round(4))

    # 2) pre-registered choice of lambda
    ref = summary.loc[0.0, "val_within_view_auc"]
    ok = summary[(summary.index > 0) & (summary.val_within_view_auc >= ref - 0.01)]
    lam = ok.val_leakage.idxmin() if len(ok) else summary.index[summary.index > 0].min()
    print("chosen lambda:", lam)

    # 3) two more seeds with the chosen lambda
    for seed in [1, 2]:
        cfg = TrainConfig(experiment=f"E4_lam{lam}", seed=seed, adv_weight=lam, **common)
        train_model(cfg, cache_path, splits, out_dir, device)